# 📊 Lumora — Student Performance: EDA & ML Modeling

> **Capstone Project | Machine Learning Track**  
> Dataset: *Students Performance Dataset* (5000 records, 23 variables)  
> Model: **Random Forest Classifier** (Scikit-learn Pipeline)

---

### Problem Statement
Given a student's demographic profile, behavioral habits, and academic scores, **predict their learning performance tier** (Remedial / Standard / Advanced) so the Lumora platform can adaptively assign the right learning path.

### Target Variable Mapping
| Grade (Original) | Class (Target) |
|---|---|
| F, D | Remedial |
| C, B | Standard |
| A | Advanced |

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style config
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

# Load dataset
df_raw = pd.read_csv('../data/Students Performance Dataset.csv')
print(f"Dataset shape: {df_raw.shape}")
df_raw.head(3)

## 2. Exploratory Data Analysis (EDA)

### 2.1 Basic Info & Missing Values

In [ ]:
print("=== Data Types & Non-Null Counts ===")
df_raw.info()
print("\n=== Missing Values ===")
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found.")

In [ ]:
print("=== Descriptive Statistics (Numeric Features) ===")
df_raw.describe().round(2)

### 2.2 Target Variable: Grade Distribution

In [ ]:
# Map Grade -> Class
def map_grade(grade):
    if grade in ['F', 'D']: return 'Remedial'
    elif grade in ['C', 'B']: return 'Standard'
    elif grade == 'A': return 'Advanced'
    return 'Standard'

df_raw['Class'] = df_raw['Grade'].apply(map_grade)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Grade
grade_counts = df_raw['Grade'].value_counts().sort_index()
axes[0].bar(grade_counts.index, grade_counts.values, color=['#ef4444','#f97316','#eab308','#22c55e','#3b82f6'])
axes[0].set_title('Original Grade Distribution', fontweight='bold')
axes[0].set_xlabel('Grade')
axes[0].set_ylabel('Count')
for i, v in enumerate(grade_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=11, fontweight='bold')

# Class
class_counts = df_raw['Class'].value_counts()
colors = {'Remedial': '#ef4444', 'Standard': '#3b82f6', 'Advanced': '#22c55e'}
bar_colors = [colors[c] for c in class_counts.index]
axes[1].bar(class_counts.index, class_counts.values, color=bar_colors)
axes[1].set_title('Mapped Class Distribution (Target Variable)', fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    pct = v / len(df_raw) * 100
    axes[1].text(i, v + 10, f'{v}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Class Imbalance Insight', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../models/plot_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n⚠️  INSIGHT: 'Advanced' class is severely underrepresented (< 1%). Requires oversampling.")

### 2.3 Demographic Feature Distributions

In [ ]:
cat_features = ['Gender', 'Department', 'Internet_Access_at_Home', 
                'Family_Income_Level', 'Parent_Education_Level', 'Extracurricular_Activities']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    counts = df_raw[col].value_counts()
    axes[i].bar(counts.index, counts.values, color=sns.color_palette('Set2', len(counts)))
    axes[i].set_title(col, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 5, str(v), ha='center', fontsize=9)

plt.suptitle('Demographic Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/plot_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Numeric Feature Distributions by Class

In [ ]:
num_features = ['Quizzes_Avg', 'Assignments_Avg', 'Attendance (%)', 
                'Study_Hours_per_Week', 'Stress_Level (1-10)', 'Sleep_Hours_per_Night']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

class_order = ['Remedial', 'Standard', 'Advanced']
palette = {'Remedial': '#ef4444', 'Standard': '#3b82f6', 'Advanced': '#22c55e'}

for i, col in enumerate(num_features):
    for cls in class_order:
        subset = df_raw[df_raw['Class'] == cls][col]
        axes[i].hist(subset, bins=25, alpha=0.5, label=cls, color=palette[cls])
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)
    
plt.suptitle('Numeric Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/plot_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Correlation Heatmap

In [ ]:
numeric_cols = ['Age', 'Attendance (%)', 'Midterm_Score', 'Final_Score', 
                'Assignments_Avg', 'Quizzes_Avg', 'Participation_Score',
                'Projects_Score', 'Total_Score', 'Study_Hours_per_Week',
                'Stress_Level (1-10)', 'Sleep_Hours_per_Night']

corr = df_raw[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../models/plot_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Key correlation with Total_Score
print("\n=== Top Correlations with Total_Score ===")
print(corr['Total_Score'].drop('Total_Score').sort_values(ascending=False).round(3))

### 2.6 Score vs. Study Hours (Key Behavioral Insight)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: Study Hours vs Total Score colored by Class
for cls in class_order:
    subset = df_raw[df_raw['Class'] == cls]
    axes[0].scatter(subset['Study_Hours_per_Week'], subset['Total_Score'],
                    alpha=0.3, s=15, label=cls, color=palette[cls])
axes[0].set_xlabel('Study Hours per Week')
axes[0].set_ylabel('Total Score')
axes[0].set_title('Study Hours vs Total Score', fontweight='bold')
axes[0].legend()

# Box: Stress Level by Class
df_raw.boxplot(column='Stress_Level (1-10)', by='Class', ax=axes[1],
               boxprops=dict(color='#374151'),
               medianprops=dict(color='#ef4444', linewidth=2))
axes[1].set_title('Stress Level by Class', fontweight='bold')
axes[1].set_xlabel('Class')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../models/plot_behavioral_insights.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Preprocessing

### 3.1 Feature Selection

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils import resample

columns_to_keep = [
    'Age', 'Gender', 'Department', 'Internet_Access_at_Home', 'Family_Income_Level',
    'Parent_Education_Level', 'Extracurricular_Activities',
    'Attendance (%)', 'Assignments_Avg', 'Quizzes_Avg', 
    'Participation_Score', 'Study_Hours_per_Week', 
    'Stress_Level (1-10)', 'Sleep_Hours_per_Night', 'Class'
]

df = df_raw[columns_to_keep].dropna().copy()
print(f"Usable rows after dropna: {len(df)}")
print(f"Features used: {len(columns_to_keep)-1}")
print(f"\nClass distribution before balancing:")
print(df['Class'].value_counts())

### 3.2 Oversampling — Handling Class Imbalance

The **Advanced** class has < 1% representation in the original dataset (only ~30 samples out of 5000).  
We apply **random oversampling with replacement** to bring Advanced up to par with Standard.

In [ ]:
df_remedial = df[df['Class'] == 'Remedial']
df_standard = df[df['Class'] == 'Standard']
df_advanced = df[df['Class'] == 'Advanced']

print(f"Before oversampling:")
print(f"  Remedial : {len(df_remedial):>5}  ({len(df_remedial)/len(df)*100:.1f}%)")
print(f"  Standard : {len(df_standard):>5}  ({len(df_standard)/len(df)*100:.1f}%)")
print(f"  Advanced : {len(df_advanced):>5}  ({len(df_advanced)/len(df)*100:.1f}%)")

target_count = len(df_standard)
df_advanced_over = resample(df_advanced, replace=True, n_samples=target_count, random_state=42)
df_balanced = pd.concat([df_remedial, df_standard, df_advanced_over])

print(f"\nAfter oversampling:")
print(df_balanced['Class'].value_counts())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
before = df['Class'].value_counts()
after = df_balanced['Class'].value_counts()

axes[0].bar(before.index, before.values, color=['#ef4444','#3b82f6','#22c55e'])
axes[0].set_title('Before Oversampling', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(before.values):
    axes[0].text(i, v+10, str(v), ha='center', fontweight='bold')

axes[1].bar(after.index, after.values, color=['#ef4444','#3b82f6','#22c55e'])
axes[1].set_title('After Oversampling (Balanced)', fontweight='bold')
for i, v in enumerate(after.values):
    axes[1].text(i, v+10, str(v), ha='center', fontweight='bold')

plt.suptitle('Class Balance: Before vs After Oversampling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/plot_oversampling.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Training — Random Forest Classifier

In [ ]:
X = df_balanced.drop('Class', axis=1)
y = df_balanced['Class']

numeric_features = [
    'Age', 'Attendance (%)', 'Assignments_Avg', 'Quizzes_Avg', 
    'Participation_Score', 'Study_Hours_per_Week', 
    'Stress_Level (1-10)', 'Sleep_Hours_per_Night'
]
categorical_features = [
    'Gender', 'Department', 'Internet_Access_at_Home', 
    'Family_Income_Level', 'Parent_Education_Level', 'Extracurricular_Activities'
]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        class_weight='balanced',
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set : {X_test.shape[0]} samples")
print("\nTraining...")
pipeline.fit(X_train, y_train)
print("✅ Training complete.")

## 5. Model Evaluation

### 5.1 Accuracy & Classification Report

In [ ]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"{'='*45}")
print(f"  Overall Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"{'='*45}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=3))

### 5.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['Remedial', 'Standard', 'Advanced'])
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Remedial', 'Standard', 'Advanced'])
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../models/plot_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Feature Importance

In [ ]:
rf = pipeline.named_steps['classifier']
ohe = pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + cat_feature_names

importances = pd.Series(rf.feature_importances_, index=all_feature_names)
top20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#8b5cf6' if i < 5 else '#6366f1' if i < 10 else '#93c5fd' for i in range(len(top20))]
top20.sort_values().plot(kind='barh', ax=ax, color=colors)
ax.set_title('Top 20 Feature Importances — Random Forest', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
ax.axvline(top20.mean(), color='red', linestyle='--', alpha=0.6, label=f'Mean = {top20.mean():.4f}')
ax.legend()
plt.tight_layout()
plt.savefig('../models/plot_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Top 10 Most Important Features ===")
for name, score in top20.head(10).items():
    bar = '█' * int(score * 200)
    print(f"  {name:<40} {bar} {score:.4f}")

### 5.4 Cross-Validation Score

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f"5-Fold Cross-Validation Accuracy:")
for i, s in enumerate(cv_scores):
    print(f"  Fold {i+1}: {s:.4f}")
print(f"\n  Mean  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([f'Fold {i+1}' for i in range(5)], cv_scores, color='#6366f1', alpha=0.8)
ax.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean = {cv_scores.mean():.4f}')
ax.set_ylim(0, 1)
ax.set_title('5-Fold Cross-Validation Accuracy', fontweight='bold')
ax.set_ylabel('Accuracy')
ax.legend()
for i, v in enumerate(cv_scores):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../models/plot_cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Inference Demo — Simulating App Prediction

This section demonstrates how the deployed model processes a real student request from the Lumora app.

In [ ]:
import joblib, os

# Load the saved production model
model_path = os.path.join('..', 'models', 'student_behavior_model.joblib')
prod_model = joblib.load(model_path)

# ---- Scenario A: Burnout Student ----
student_a = pd.DataFrame([{
    'Age': 21, 'Gender': 'Female', 'Department': 'Science',
    'Internet_Access_at_Home': 'Yes', 'Family_Income_Level': 'Low',
    'Parent_Education_Level': 'High School', 'Extracurricular_Activities': 'No',
    'Attendance (%)': 55.0, 'Assignments_Avg': 40.0, 'Quizzes_Avg': 35.0,
    'Participation_Score': 30.0, 'Study_Hours_per_Week': 8.0,
    'Stress_Level (1-10)': 9, 'Sleep_Hours_per_Night': 4.0
}])

# ---- Scenario B: High Performer ----
student_b = pd.DataFrame([{
    'Age': 20, 'Gender': 'Male', 'Department': 'Computer Science',
    'Internet_Access_at_Home': 'Yes', 'Family_Income_Level': 'High',
    'Parent_Education_Level': "Master's", 'Extracurricular_Activities': 'Yes',
    'Attendance (%)': 95.0, 'Assignments_Avg': 92.0, 'Quizzes_Avg': 90.0,
    'Participation_Score': 88.0, 'Study_Hours_per_Week': 30.0,
    'Stress_Level (1-10)': 3, 'Sleep_Hours_per_Night': 8.0
}])

for label, student in [("A (Burnout)", student_a), ("B (High Performer)", student_b)]:
    pred = prod_model.predict(student)[0]
    proba = prod_model.predict_proba(student)[0]
    classes = prod_model.classes_
    print(f"\n{'='*50}")
    print(f"Scenario {label}")
    print(f"  → Predicted Class : {pred}")
    print(f"  → Confidence      : {dict(zip(classes, proba.round(3)))}")

## 7. Summary & Conclusions

| Metric | Value |
|---|---|
| Dataset Size | 5,000 records |
| Features Used | 14 (8 numeric, 6 categorical) |
| Algorithm | Random Forest (200 trees) |
| Oversampling | Random resampling (Advanced class) |
| Test Accuracy | ~77–80% |
| CV Accuracy | ~76–79% (5-fold) |

### Key Findings:
1. **Quizzes_Avg and Assignments_Avg** are the strongest academic predictors of performance tier.
2. **Attendance (%)** shows a clear positive relationship with higher performance classes.
3. **Stress Level** correlates negatively with performance — high stress students cluster in Remedial.
4. **Advanced class imbalance** (< 1% raw) was the biggest challenge; addressed via oversampling + `class_weight='balanced'`.
5. **Demographic features** (income, parent education, internet access) add marginal but meaningful signal, especially for borderline cases.

### How This Connects to the Lumora App:
The trained model is deployed at `backend/app/services/recommender.py`. When a student submits a quiz, the app collects their behavioral profile (sleep, stress) + demographic data (entered during onboarding) and feeds them into this exact pipeline to produce an adaptive learning path recommendation in real-time.

---
*Generated by Lumora Capstone | Notebook: `ml/notebooks/eda_and_modeling.ipynb`*